# 5. Effects of Different Paths

In this notebook, we test the path-following capabilities on different paths.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using LinkedMasses

## Paths

For a fair comparison, each path starts at the same point: $[2, 2]$.

### Circular Parh

The original circular path:

In [ ]:
function path_circle(s::Real)    
    R = 10.0
    ϕ0 = -π / 2
    p_origin = [2.0, 2.0 + R]
    return p_origin + R * [cos(ϕ0 + s), sin(ϕ0 + s)]
end

### Big Circle

Let's create a circle of radius 20:

In [ ]:
function path_circle_big(s::Real)    
    R = 20.0
    ϕ0 = -π / 2
    p_origin = [2.0, 2.0 + R]
    return p_origin + R * [cos(ϕ0 + s), sin(ϕ0 + s)]
end

### Sine Wave

Sine wave with amplitude $10$ and wavenumber $50$:

In [ ]:
function path_sine(s::Real)
    A = 10.0
    L = 50.0
    ϕ0 = -π / 2
    return [s + 2.0, A * sin(2π * (s / L) + ϕ0) + 12.0]
end

### Straight Line

Finally, a straight line:

In [ ]:
path_line(s::Real) = [s + 2.0, 2.0]

## Path Curvature

In [ ]:
import ForwardDiff

function curvature(path_fcn::Function, s::Real)
    ∂p_fcn = (_s) -> ForwardDiff.derivative(path_fcn, _s)
    dp = ∂p_fcn(s)
    ddp = ForwardDiff.derivative(∂p_fcn, s)
    return (dp[1] * ddp[2] - dp[2] * ddp[1]) / (dp[1]^2 + dp[2]^2)^(3/2)
end

s_circle = range(0, stop=2π, length=101)
k_circle = [curvature(path_circle, s) for s in s_circle]
k_circle_big = [curvature(path_circle_big, s) for s in s_circle]
s_sine = range(0, stop=50, length=101)
k_sine = [curvature(path_sine, s) for s in s_sine]
s_normalized = range(0, stop=1, length=101)
k_line = [curvature(path_line, s) for s in s_normalized]

using CSV, DataFrames
df_curvatures = DataFrame(s=s_sine,
                          k_circle=k_circle,
                          k_circle_big=k_circle_big,
                          k_sine=k_sine,
                          k_line=k_line)
folder = joinpath(@__DIR__, "csv")
mkpath(folder)
CSV.write(joinpath(folder, "curvatures.csv"), df_curvatures)

using Plots
plot(s_normalized, k_circle, label="Circle R=10", xlabel="s", ylabel="Curvature", legend=:topright)
plot!(s_normalized, k_circle_big, label="Circle R=20")
plot!(s_normalized, k_sine, label="Sine Path")
plot!(s_normalized, k_line, label="Straight Line")

## Controllers

In [ ]:
V_c = [-0.2, 0.0]

m0 = 200.0
m = 40.0
c0 = 250.0
c = 50.0
L = 10.0

params = LinkedMassesParameters(L, m0, m, c0, c)

ε = 0.7

U = 1.0
Δ = 5.0
los = LOSParameters(U, Δ)

k_v = 0.1
γ = 5.0

paths = [
    path_circle,
    path_circle_big,
    path_sine,
    path_line
]

labels = [
    "circle",
    "big_circle",
    "sine",
    "line"
]

ctrls = [
    AdaptiveLinearizingController(params, los, path, k_v, γ, ε, V_c) for path in paths
]

In [ ]:
k_err = 1.5
params_estimate = LinkedMassesParameters(L, m0, m, k_err*c0, c/k_err)
ζ_hat0 = parameter_vector(params_estimate, ε, zeros(2))

using DifferentialEquations
T_stop = 300

# Initial state: both masses at rest, first mass at origin, second mass east of it
p0_init = [0.0, 0.0]
θ_init = -π / 2
v0_init = [0.0, 0.0]
ω_init = 0.0
s_init = 0.0
init_state = AdaptiveSimulationState(p0_init, θ_init, v0_init, ω_init, s_init, ζ_hat0)

sols = [
    simulate(ctrl, init_state, T_stop; reltol=1e-6, saveat=2.0) for ctrl in ctrls
]

In [ ]:
using Plots, ColorSchemes, LaTeXStrings
using CSV, DataFrames

scheme = ColorSchemes.Dark2_5.colors

fig1 = plot(title="Path-Following Error", xlabel="Time (s)", ylabel="Error (m)")
fig2 = plot(title="Velocity Error", xlabel="Time (s)", ylabel="Error (m/s)")

df = DataFrame(time = sols[1].t)
for i in eachindex(labels)
    label = labels[i]
    sol = sols[i]
    ctrl = ctrls[i]
    e_x_i, e_y_i, v_x_i, v_y_i = get_errors(sol, ctrl)
    df[!, Symbol("e_x_$label")] = e_x_i
    df[!, Symbol("e_y_$label")] = e_y_i
    df[!, Symbol("v_x_$label")] = v_x_i
    df[!, Symbol("v_y_$label")] = v_y_i

    plot!(fig1, sol.t, e_x_i, label=label, color=scheme[i])
    plot!(fig1, sol.t, e_y_i, label="", linestyle=:dash, color=scheme[i])
    plot!(fig2, sol.t, v_x_i, label=label, color=scheme[i])
    plot!(fig2, sol.t, v_y_i, label="", linestyle=:dash, color=scheme[i])
end
    
CSV.write(joinpath(folder, "path_comparison_errors.csv"), df)

plot(fig1, fig2, layout=(2,1), size=(800,600))
